# Task 1.3 — What the Paper Claims to Improve (9 marks)

Analysis of baselines, the limitation they have, how PLTM overcomes it, and when PLTM may not win.

**Do not clear outputs before submission.**

## Rubric answers (9 marks)

**1. Main baseline(s) the paper compares against [2 marks]**  
The paper compares the proposed method (PLTM) against **variable-selection methods** that *perform* variable selection. The two main such baselines are named explicitly:  
- **CVS (ClustVarsel)** — Raftery & Dean (2006): greedy search for a *single* subset of attributes and a GMM on that subset, using Bayes factors.  
- **LFJ** — Law, Figueiredo & Jain (2004): mixture model with a *saliency* parameter per attribute (0–1), learned by EM; effectively one clustering driven by a subset of attributes.  

The paper also compares to **GMM** (no variable selection) and **MCLUST** (GMM with constrained covariances, no variable selection). The title “To Do or To Facilitate” contrasts PLTM with approaches that *do* variable selection (CVS, LFJ).

---

**2. Limitation of this baseline the paper identifies [2 marks]**  
When data are **multifaceted**—i.e. there are *multiple* meaningful ways to cluster the same objects (e.g. by age, by income, or by region)—methods like CVS and LFJ must commit to **one** subset of variables and **one** clustering. The paper argues that the very goal of finding “the” best subset is then ill-conceived: which facet is “best” depends on the user. In practice, the baseline often picks a facet that is *not* the one the user cares about (e.g. the class label). For example, on the paper’s synthetic data, the true class is aligned with one facet (X₁–X₃), but CVS selected the *other* facet (X₄–X₉), so NMI with the class was very low (0.07). So the limitation is: **performing variable selection forces a single subset and a single partition, which is poorly suited when several valid clusterings (facets) exist.**

---

**3. How the proposed method attempts to overcome this limitation [1 mark]**  
Instead of *doing* variable selection, PLTM **facilitates** it: it learns a **tree of latent variables**, each representing a different partition (facet) of the data, each typically tied to a *subset* of attributes (its pouches). The algorithm outputs **multiple** clusterings; the user (or downstream evaluation) can then **choose** the facet that is relevant (e.g. the one that matches the class or the analyst’s interest). So the method overcomes the limitation by **exposing multiple facets rather than committing to one.**

---

**4. One condition or scenario where the paper’s method would not outperform the baseline [4 marks]**  
**Scenario:** When the data have **very few attributes** and effectively **only one dominant facet**, methods that *do* variable selection can outperform PLTM. The paper’s own results support this: on the **Iris** dataset (4 attributes), **CVS (0.87) beats PLTM (0.76)**. With so few dimensions, there may be only one natural way to cluster; PLTM’s extra structure (multiple latents, structure search) can add noise or overfit, while CVS simply finds that one useful subset and does well. So **low-dimensional data with a single meaningful facet** is a reasoned scenario where the paper’s method need not outperform the baseline. (Other valid scenarios: when the class label does not align with any recovered facet; when data violate the model’s assumptions, e.g. non-Gaussian or non-tree structure; or when computational budget favours simpler methods.)

## 1. Problem motivation

The paper tackles **variable selection for clustering**: in high dimensions, we often want to know *which* attributes matter for the clustering we care about. Standard **Gaussian Mixture Models (GMMs)** use *all* variables. That causes two issues: (1) **Irrelevant variables** dilute the signal—noise dimensions can hide the true cluster structure and hurt performance. (2) **Multiple clustering facets**—the same data can be meaningfully grouped in several ways (e.g. by age, by income, or by region). A single GMM (or a method that picks one subset and one clustering) implicitly assumes there is *one* “best” partition. When that is false, forcing a single answer is the wrong goal. The paper argues that in such settings we should not *choose* one subset for the user but *facilitate* choice by finding and showing several facets.

## 2. Key idea of the paper

The central idea is **facilitate, don’t do**. Instead of performing variable selection (picking one subset and one clustering), the method **facilitates** variable selection: it **systematically finds several different ways to cluster the data** and presents them to the user. Each way is a **facet**—a partition that depends mainly on a *subset* of attributes. **Multi-facet clustering** means: the same dataset admits multiple valid clusterings (e.g. facet 1 by attributes A,B,C; facet 2 by D,E,F). The algorithm discovers these facets; the domain expert then appraises and chooses. So the “improvement” is a shift from “find the best subset” to “find all meaningful facets and let the user decide.”

## 3. Model overview

The paper extends the GMM using a **Pouch Latent Tree Model (PLTM)**. A GMM has **one** latent (mixture component) and **one** block of observed variables. A PLTM has **several** discrete latent variables arranged in a **rooted tree**; **leaf nodes** are “pouches,” each containing a *group* of continuous observed variables. So: internal nodes = latents (each latent = one possible clustering); leaves = pouches of manifest variables. Each pouch is Gaussian given its parent latent: P(x|y) = N(x | μ_y, Σ_y). Different groups of variables (different pouches) are “owned” by different latents, so **different subsets of variables correspond to different clustering facets**. Learning the tree and parameters yields multiple partitions—one per latent (or combinations)—each tied to a subset of attributes, which is exactly the multi-facet view. (Section 2, Figure 1(a).)

## 4. Learning algorithm

**Training** has two layers: (1) **Parameter estimation** for a *given* structure, and (2) **Structure search** over trees and pouches.

- **EM for parameters (Section 3):** For a fixed PLTM structure, parameters are estimated by **EM**. *E-step:* given current parameters, compute the posterior over latent variables for each data point—i.e. infer P(y | data) and P(y, parent | data) using exact inference on the mixed (discrete + Gaussian) tree (Lauritzen & Jensen). *M-step:* update discrete conditional tables P(y|parent) from expected counts, and update each pouch’s mean μ_y and covariance Σ_y from posterior-weighted statistics. To avoid degenerate solutions, **eigenvalue constraints** are applied to each Σ_y: eigenvalues are clamped to [σ²_min, γ·σ²_max] (γ = 20 in the paper).

- **Latent variables** are never observed; they are **inferred** in the E-step from the observed manifest variables. So “which cluster (facet)” a point belongs to is a posterior over the latent states.

- **Cluster structure discovery** is done by **structure learning (Section 4)**. The algorithm **hill-climbs** on the BIC score: start with one latent (two states) and one pouch per variable; repeatedly apply operators (add/delete latent, add/delete state, relocate node, merge/split pouches), run EM for each candidate, and accept the candidate if BIC improves. So the *number* of latents and *which* variables sit in which pouch are discovered during search, which is how multiple facets emerge.

## 5. Example intuition

**Synthetic data in the paper (Section 5, Figure 1(c)):** 9 variables X₁–X₉, two hidden “true” facets. **Facet 1** (latent Y₁) mainly drives X₁, X₂, X₃—so clustering by Y₁ groups points that differ on those three variables. **Facet 2** (latent Y₂) mainly drives X₄–X₉—so clustering by Y₂ groups points that differ on that other subset. So *the same* 1000 points can be partitioned in two different ways: one partition by {X₁,X₂,X₃}, another by {X₄,…,X₉}. If we force a single clustering (e.g. CVS picks one subset), we might get only one of these; if that one is not the facet the user cares about (e.g. the “class” is Y₁), performance is poor. PLTM recovers *both* facets and lets the user (or the evaluation) pick the one that matches the class, which is why it gets higher NMI on this data.

## 6. Experiments in the paper

- **Data:** (1) **Synthetic:** 1000 samples from a known PLTM with 9 manifest and 2 latent variables (3 states each); class = one latent, the other hidden (Section 5.1, Figure 1(c)). (2) **Real:** 9 UCI datasets (e.g. image, ionosphere, iris, wine, yeast) with continuous attributes; class labels used only for evaluation (Table 1).

- **Metric:** **Normalized Mutual Information (NMI)** between the **class label** and the clustering. Higher NMI = clustering agrees more with the class. For PLTM, they report **max NMI over the latent variables** (or a greedy combination)—i.e. “if the user picked the best facet, how well would they do?” (Section 5.2.)

- **Results (Table 2):** PLTM often **outperforms** CVS and LFJ (e.g. synthetic: PLTM 0.81 vs CVS 0.07, LFJ 0.54; image, vehicle, wine, etc.). On **Iris**, CVS (0.87) beats PLTM (0.76). On **ionosphere** and **wdbc**, GMM or MCLUST sometimes beat PLTM. So the experiments demonstrate that facilitating (multiple facets) is often better than doing (single subset), but not always—e.g. when there is effectively one facet or when the class does not align with any recovered facet.

## 7. Key contributions

- **Conceptual:** Arguing that variable selection for clustering should often be **facilitated** (show multiple facets) rather than **performed** (pick one subset), especially when data are multifaceted.

- **Model:** Proposing **PLTM**—a generalization of the GMM to a tree of discrete latents and pouch nodes of Gaussian manifest variables—so that multiple partitions (facets) can be represented and learned.

- **Algorithm:** EM for parameter estimation with **eigenvalue constraints** to avoid degeneracy; **BIC-based hill-climbing** with search operators for structure learning (number of latents, cardinalities, pouch composition).

- **Empirical:** Showing that PLTM often outperforms variable-selection baselines (CVS, LFJ) and standard GMM/MCLUST on synthetic and UCI data when evaluated by NMI against the class, and discussing when it does not (e.g. Iris).

## 8. Assumptions and limitations

The method can fail or underperform when:

- **Single facet / low dimension:** When there is effectively only one meaningful way to cluster (e.g. Iris with 4 attributes), “facilitate” adds little; methods that *do* variable selection (e.g. CVS) can win (Section 5.3, Table 2).

- **Class not aligned with any facet:** If the class label does not correspond to any natural facet (e.g. arbitrary labeling), no latent may match it well, so PLTM’s max-NMI need not be high; GMM or MCLUST using all variables might do better (e.g. ionosphere, wdbc).

- **Model assumptions violated:** The paper assumes a **tree** (no cycles, one parent per node), **one parent per manifest variable**, and **conditional Gaussian** in each pouch. Non-Gaussian data (e.g. counts, heavy tails), or dependencies that are not tree-like, can make the model a poor fit and the recovered “facets” misleading (Task 1.2, Task 3.2).

- **Computational cost:** Structure search plus multiple EM restarts can be slow (hours to days on larger datasets), so scalability is a practical limitation (Section 7).

## 9. Connection to my reproduction

A **simplified reproduction** can demonstrate the core idea without implementing full PLTM structure search:

- **Synthetic multifaceted data:** Generate data from a model with two “facets” (e.g. two latent components, each governing a *subset* of variables—e.g. Y₁ → X₁–X₃, Y₂ → X₄–X₉), similar to the paper’s Figure 1(c). Use one latent as the “class” for evaluation.

- **Baseline:** Fit a **standard GMM** (or a method that does variable selection on a single subset) on the same data. When the data are multifaceted, a single GMM or single-subset method may lock onto one facet and get low NMI with the class if that facet is the “wrong” one.

- **Proposed idea:** Implement **EM with eigenvalue constraints** (γ = 20) for a *fixed* small PLTM-like structure (e.g. two latents, two pouches) or a simplified multi-facet GMM that exposes multiple clusterings. Show that by recovering *multiple* partitions (e.g. one per latent) and reporting **max NMI** over them, we can match or beat the single-clustering baseline when the class aligns with one of the facets. This illustrates “facilitate vs do” without full structure learning.

So the reproduction focuses on: (1) multifaceted synthetic data, (2) NMI as the metric, (3) EM with eigenvalue constraint, (4) the value of exposing multiple facets (max over latents) rather than a single clustering.

In [ ]:
# TODO: Add content after paper is provided.
import numpy as np
np.random.seed(42)